# Group-wise comparisons

Compare the relative abundance and protein signatures of pancreatic cell types across donors and disease groups. Each section contributes one donor observation.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/04_group_comparisons.ipynb).

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "pancreas_islets.h5ad").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the processed PhenoCycler data directory.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})
DONOR = "6533"                      # Any donor listed in the manifest
DISEASE = "T1D"                     # ND, AAb+, or T1D
LEVEL = "primary"                    # primary, detail, immune_class, t_subtype, macrophage_profile
MARKERS = ["GCG","INS","SST","CD99","CD3e","CD20","CD68","CD31","EpCAM"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
type_order = ["Alpha","Beta","Delta","Endocrine (ambiguous subtype)","Endocrine (unresolved subtype)",
              "Exocrine","Immune","Blood vessel","Lymphatic","Nerve","Unresolved (no lineage evidence)"]
inks = {"Alpha":"#cf6c30", "Beta":"#287cac", "Delta":"#4c8b58", "Immune":"#965084",
        "Exocrine":"#c1b7a6", "Endocrine (ambiguous subtype)":"#bda559",
        "Endocrine (unresolved subtype)":"#99853f", "Blood vessel":"#b34b50",
        "Lymphatic":"#68a69a", "Nerve":"#5b558c", "Unresolved (no lineage evidence)":"#868686"}
for label in type_order:
    inks.setdefault(str(label), "#73777d")

## 1. Cell-type composition by donor

Each bar partitions all admitted cells. Short bars within the distributions below mark the group median.

In [ ]:
primary = counts.loc[counts.level.eq("primary")]
wide = primary.pivot_table(index=["disease","donor"], columns="label", values="cells",
                           aggfunc="sum", fill_value=0, observed=True)
assert (wide.sum(axis=1) == primary.groupby(["disease","donor"], observed=True).eligible_cells.first().reindex(wide.index)).all()
frac = wide.div(wide.sum(axis=1), axis=0)
sort = sorted(frac.index, key=lambda pair: (group_order.index(pair[0]), int(pair[1])))
frac = frac.loc[sort]
fig, ax = plt.subplots(figsize=(12,4.9))
bottom = np.zeros(len(frac))
for label in frac.columns:
    ax.bar(np.arange(len(frac)), 100*frac[label], bottom=bottom, width=.83, color=inks[label], label=label)
    bottom += 100*frac[label].to_numpy()
ax.set(xticks=range(len(frac)), xticklabels=frac.index.get_level_values("donor"), ylim=(0,100),
       ylabel="Admitted cells (%)", xlabel="Pancreas donor", xlim=(-.7,len(frac)-.3))
ax.tick_params(axis="x", rotation=60)
for disease in group_order:
    loc = np.flatnonzero(frac.index.get_level_values("disease") == disease)
    ax.text(loc.mean(),103,disease,ha="center",color=group_ink[disease],weight="bold",size=9)
ax.legend(loc="upper center",bbox_to_anchor=(.5,-.26),ncol=4,frameon=False,fontsize=8)
fig.subplots_adjust(bottom=.33)
plt.show()
display(pd.DataFrame({"donor cells":wide.xs(DONOR,level="donor").iloc[0],
                      "donor %":100*frac.xs(DONOR,level="donor").iloc[0],
                      f"{DISEASE} mean donor %":100*frac.xs(DISEASE).mean()}).round(3))

In [ ]:
# Disease-group means are computed after each donor's fractions.
cohort = counts[counts.level.eq("primary")]
frac_by_donor = cohort.pivot_table(index=["donor","disease"],columns="label",values="fraction_tissue",
                                   aggfunc="first",observed=True).reindex(columns=type_order).fillna(0)
assert np.allclose(frac_by_donor.sum(axis=1), 1.0)
panel = ["Alpha","Beta","Delta","Immune"]
fig, axes = plt.subplots(1,len(panel),figsize=(12,3.7))
for ax,label in zip(axes,panel):
    for i,g in enumerate(group_order):
        v = (100*frac_by_donor.xs(g,level="disease")[label]).sort_index()
        ax.scatter(i+np.linspace(-.15,.15,len(v)), v.values, s=24, color=group_ink[g],alpha=.85)
        ax.hlines(v.median(), i-.24, i+.24, color="#20282b", linewidth=1.6)
    ax.set(xticks=range(3),xticklabels=group_order,title=label,ylim=(0,None))
axes[0].set_ylabel("Admitted cells (%)")
fig.tight_layout()
plt.show()

## 2. Compare disease-group marker expression

Each panel uses the same marker scale. Color is mean protein expression; dot area is the fraction of measured cells at or above 0.10. Donors contribute equally within each disease group. A gray × marks missing measurements.

In [ ]:
def protein_dotplot(group=None, donor_id=None):
    eligible = counts.loc[counts.level.eq(LEVEL), ["donor","disease","label","comparable","assessable"]]
    frame = evidence.loc[evidence.level.eq(LEVEL) & evidence.marker.isin(MARKERS)].merge(
        eligible, on=["donor","disease","label"], validate="many_to_one")
    frame = frame.loc[frame.donor.eq(donor_id) if donor_id is not None else frame.disease.eq(group)].copy()
    markers = [m for m in MARKERS if m in evidence.marker.unique()]
    reportable = frame.assessable if donor_id is not None else frame.comparable
    frame = frame.loc[reportable & frame.n_measured.gt(0)]
    if frame.empty:
        print(f"No measured cells at {LEVEL} for this selection.")
        return
    labels = sorted(frame.label.unique(), key=lambda s: (type_order.index(s) if s in type_order else len(type_order), s))
    def matrix(column):
        return frame.pivot_table(index="label",columns="marker",values=column,aggfunc="mean",
                                 observed=True).reindex(index=labels,columns=markers)
    mean, fraction = matrix("mean_scaled"), matrix("positive_fraction")
    # Scanpy accepts precomputed sizes and colors. These summaries preserve cell and donor denominators.
    summary = ad.AnnData(mean.to_numpy(), obs=pd.DataFrame({"cell_type": pd.Categorical(labels,labels)},index=labels),
                         var=pd.DataFrame(index=markers))
    scope = f"Donor {donor_id}" if donor_id is not None else f"{group} · {frame.donor.nunique()} donors"
    plot = sc.pl.DotPlot(summary, markers, groupby="cell_type", dot_color_df=mean,
                         dot_size_df=fraction, vmin=0, vmax=1, title=f"{scope} · {LEVEL}",
                         figsize=(8.5, max(3.6, len(labels)*.28+1.8)))
    plot.style(cmap="viridis", dot_min=0, dot_max=1, largest_dot=150).legend(
        colorbar_title="Mean scaled protein", size_title="Positive cells (%)")
    plot.make_figure()
    ax = plot.get_axes()["mainplot_ax"]
    y, x = np.where(fraction.isna().to_numpy())
    ax.scatter(x+.5, y+.5, marker="x", s=16, linewidth=.8, color="#747474")
    plt.show()


for group in group_order:
    protein_dotplot(group=group)

## 3. Between-group protein differences

Each row is one donor’s mean expression within the selected phenotype. The heatmap lets you see donor-to-donor variability behind a group average.

In [ ]:
PHENOTYPE = "Beta"
keys = counts.loc[counts.level.eq(LEVEL) & counts.label.eq(PHENOTYPE),["donor","comparable"]]
measured = evidence.loc[evidence.level.eq(LEVEL) & evidence.label.eq(PHENOTYPE)].merge(keys,on="donor",validate="many_to_one")
z = measured.loc[measured.comparable & measured.n_measured.gt(0) & measured.marker.isin(MARKERS)]
means = z.pivot_table(index=["disease","donor"],columns="marker",values="mean_scaled",observed=True)
if len(means):
    means = means.loc[sorted(means.index,key=lambda k:(group_order.index(k[0]),int(k[1])))]
    means = means.reindex(columns=[m for m in MARKERS if m in means])
    obs=pd.DataFrame({"section":[f"{g} · {d}" for g,d in means.index]},index=means.index.get_level_values("donor"))
    obs["section"]=pd.Categorical(obs.section,categories=obs.section)
    m=ad.AnnData(means.to_numpy(),obs=obs,var=pd.DataFrame(index=means.columns))
    cmap=plt.get_cmap("viridis").copy();cmap.set_bad("#d9d9d9")
    sc.pl.heatmap(m,list(m.var_names),groupby="section",swap_axes=True,show_gene_labels=True,cmap=cmap,
                  vmin=0,vmax=1,figsize=(max(9,.45*len(obs)),4))
else:
    print(f"No comparable {PHENOTYPE} cells at {LEVEL}.")